In [ ]:
import os
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
import cv2
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Đang sử dụng thiết bị: {device}")

def load_ben_color(path, sigmaX=10):
    image = cv2.imread(path)
    if image is None: # Cẩn thận thêm bước check ảnh lỗi hỏng
        raise ValueError(f"Không thể đọc file ảnh (file có thể bị hỏng): {path}")
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = cv2.resize(image, (256, 256))
    image = cv2.addWeighted(image, 4, cv2.GaussianBlur(image, (0,0), sigmaX), -4, 128)
    return Image.fromarray(image)

Đang sử dụng thiết bị: cuda


In [ ]:
class RobustEyePacsDataset(Dataset):
    def __init__(self, csv_file, root_dir, transform=None):
        self.data = pd.read_csv(csv_file)
        self.root_dir = root_dir
        self.transform = transform

        # 1. Quét toàn bộ ổ cứng để lập bản đồ đường dẫn ảnh
        self.image_paths = {}
        print("Đang quét toàn bộ thư mục để tìm ảnh. Vui lòng đợi...")

        for subdir, dirs, files in os.walk(root_dir):
            for file in files:
                # Tìm tất cả các file có đuôi ảnh hợp lệ
                if file.lower().endswith(('.jpeg', '.jpg', '.png')):
                    # Lấy tên gốc của ảnh (Ví dụ: '12392_right')
                    img_id = os.path.splitext(file)[0]
                    # Lưu đường dẫn tuyệt đối của nó vào từ điển
                    self.image_paths[img_id] = os.path.join(subdir, file)

        print(f"Đã quét xong! Tìm thấy tổng cộng {len(self.image_paths)} ảnh trên ổ đĩa.")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        # 1. Lấy chuỗi từ file CSV và gọt sạch dấu cách thừa
        raw_img_string = str(self.data.iloc[idx, 0]).strip()

        # 2. XÓA BỎ ĐUÔI FILE (nếu có) để chỉ lấy phần tên gốc
        # Ví dụ: '12472_left.jpeg' -> '12472_left'
        img_id = os.path.splitext(raw_img_string)[0]

        # 3. Kiểm tra xem ảnh này có tồn tại trong bản đồ đã quét không
        if img_id not in self.image_paths:
            raise FileNotFoundError(
                f"LỖI CHÍNH XÁC: Nhãn '{img_id}' (đã gọt từ '{raw_img_string}') có trong file CSV "
                f"nhưng KHÔNG THỂ TÌM THẤY file ảnh ở bất kỳ thư mục con nào trong '{self.root_dir}'."
            )

        # Lấy đường dẫn chuẩn xác
        img_path = self.image_paths[img_id]

        # Tiền xử lý
        image = load_ben_color(img_path)
        label = int(self.data.iloc[idx, 1])

        if self.transform:
            image = self.transform(image)

        return image, label

# --- KHỞI TẠO DATALOADER ---
data_transforms = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Trỏ tới thư mục dataset tổng. Lớp RobustDataset sẽ tự lo việc lặn vào các thư mục con
dataset = RobustEyePacsDataset(
    csv_file='/content/dataset/labels_6k.csv',
    root_dir='/content/dataset/',
    transform=data_transforms
)

dataloader = DataLoader(dataset, batch_size=32, shuffle=True)
print("Đã khởi tạo xong Dataset với tính năng tự động gọt đuôi file!")

🔍 Đang quét toàn bộ thư mục để tìm ảnh. Vui lòng đợi...
✅ Đã quét xong! Tìm thấy tổng cộng 6000 ảnh trên ổ đĩa.
✅ Đã khởi tạo xong Dataset với tính năng tự động gọt đuôi file!


In [ ]:
from sklearn.metrics import accuracy_score, cohen_kappa_score
import numpy as np

# ... (Khai báo mô hình, loss, optimizer giữ nguyên như cũ) ...

num_epochs = 10
save_path = '/content/drive/MyDrive/resnet50_eyepacs_backup.pth'

print("Bắt đầu quá trình huấn luyện có Tracking Metrics...")
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    # Tạo danh sách để hứng kết quả dự đoán của toàn bộ epoch
    all_preds = []
    all_labels = []

    for i, (images, labels) in enumerate(dataloader):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        # --- THÊM PHẦN TÍNH METRICS ---
        # Lấy class có xác suất cao nhất làm kết quả dự đoán
        _, preds = torch.max(outputs, 1)

        # Đẩy kết quả từ GPU về CPU và chuyển thành Numpy array để sklearn đọc được
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        # ------------------------------

        if (i+1) % 50 == 0:
            print(f"  - Đang train Epoch [{epoch+1}/{num_epochs}], Batch [{i+1}/{len(dataloader)}]")

    # Tính toán Metrics tổng kết cho cả Epoch
    epoch_loss = running_loss / len(dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)

    # Tính QWK (Sử dụng tham số weights='quadratic')
    epoch_qwk = cohen_kappa_score(all_labels, all_preds, weights='quadratic')

    print(f'KẾT QUẢ EPOCH {epoch+1}/{num_epochs}:')
    print(f'   - Loss: {epoch_loss:.4f}')
    print(f'   - Accuracy: {epoch_acc:.4f} ({(epoch_acc*100):.2f}%)')
    print(f'   - Điểm QWK: {epoch_qwk:.4f} (Càng gần 1.0 càng tốt)')

    torch.save(model.state_dict(), save_path)
    print(f"  Đã lưu backup tự động vào Drive!\n")

print("ĐÃ HOÀN THÀNH TRAINING TOÀN BỘ!")

🚀 Bắt đầu quá trình huấn luyện có Tracking Metrics...
  - Đang train Epoch [1/10], Batch [50/188]
  - Đang train Epoch [1/10], Batch [100/188]
  - Đang train Epoch [1/10], Batch [150/188]
🔥 KẾT QUẢ EPOCH 1/10:
   - Loss: 0.6515
   - Accuracy: 0.7765 (77.65%)
   - Điểm QWK: 0.5756 (Càng gần 1.0 càng tốt)
  💾 Đã lưu backup tự động vào Drive!

  - Đang train Epoch [2/10], Batch [50/188]
  - Đang train Epoch [2/10], Batch [100/188]
  - Đang train Epoch [2/10], Batch [150/188]
🔥 KẾT QUẢ EPOCH 2/10:
   - Loss: 0.6055
   - Accuracy: 0.7872 (78.72%)
   - Điểm QWK: 0.6214 (Càng gần 1.0 càng tốt)
  💾 Đã lưu backup tự động vào Drive!

  - Đang train Epoch [3/10], Batch [50/188]
  - Đang train Epoch [3/10], Batch [100/188]
  - Đang train Epoch [3/10], Batch [150/188]
🔥 KẾT QUẢ EPOCH 3/10:
   - Loss: 0.5775
   - Accuracy: 0.7977 (79.77%)
   - Điểm QWK: 0.6515 (Càng gần 1.0 càng tốt)
  💾 Đã lưu backup tự động vào Drive!

  - Đang train Epoch [4/10], Batch [50/188]
  - Đang train Epoch [4/10], Batch 